In [4]:
import cv2
import numpy as np
import plotly.express as px

# 1. Đọc ảnh
img = cv2.imread('screenshot.png')
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

# 2. Phân ngưỡng (Threshold)
# Bàn cờ sáng màu, nền tối -> Cài ngưỡng khoảng 150. 
# Những pixel > 150 sẽ thành 255 (Trắng), ngược lại thành 0 (Đen)
ret, thresh = cv2.threshold(gray, 150, 255, cv2.THRESH_BINARY)

# 3. Tìm contours (đường bao)
# cv2.RETR_EXTERNAL: Chỉ lấy đường bao ngoài cùng
contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

if contours:
    # 4. Tìm contour có diện tích lớn nhất (chính là bàn cờ)
    board_contour = max(contours, key=cv2.contourArea)
    
    # Lấy tọa độ hình chữ nhật bao quanh contour đó
    x, y, w, h = cv2.boundingRect(board_contour)
    
    print(f"Tìm thấy bàn cờ tại: Tọa độ X={x}, Y={y}, Rộng={w}, Cao={h}")

    # --- BƯỚC TÙY CHỌN: Vẽ khung xanh lá cây để kiểm tra ---
    img_debug = img.copy()
    cv2.rectangle(img_debug, (x, y), (x+w, y+h), (0, 255, 0), 3)
    
    # 5. Cắt (Crop) bàn cờ ra khỏi ảnh gốc bằng Numpy slicing
    board_crop = img[y:y+h, x:x+w]

    # --- HIỂN THỊ KẾT QUẢ BẰNG PLOTLY ---
    # Đổi BGR sang RGB cho Plotly
    img_debug_rgb = cv2.cvtColor(img_debug, cv2.COLOR_BGR2RGB)
    board_crop_rgb = cv2.cvtColor(board_crop, cv2.COLOR_BGR2RGB)

    # Hiện ảnh gốc có vẽ khung
    fig1 = px.imshow(img_debug_rgb, title="Detected Board (Green Box)")
    fig1.update_layout(margin=dict(l=0, r=0, b=0, t=30))
    fig1.show()

    # Hiện ảnh bàn cờ đã được cắt
    fig2 = px.imshow(board_crop_rgb, title="Extracted Board")
    fig2.update_layout(margin=dict(l=0, r=0, b=0, t=30))
    fig2.show()
    
else:
    print("Không tìm thấy bàn cờ nào!")


Tìm thấy bàn cờ tại: Tọa độ X=0, Y=81, Rộng=700, Cao=971


In [5]:
# Thay thế bước 2 bằng:
edges = cv2.Canny(gray, 50, 150)
# Ở bước 3, tìm contour trên biến `edges` thay vì `thresh`
contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
